In [2]:
import pandas as pd
import numpy as np
import datetime as dt
#import random
#import os
import matplotlib.pyplot as plt
#import seaborn as sns
#import requests

from datetime import datetime, timedelta

from core.data import load_from_kaggle

In [3]:
dataset_link = "nokkyu/deutsche-bahn-db-delays" # replace with your dataset link from Kaggle 
destination = "../data/raw"
dataset_name = dataset_link.split("/")[-1]

files = load_from_kaggle(
    dataset_link=dataset_link, 
    destination=destination,
    )

Destination directory '../data/raw/deutsche-bahn-db-delays' already exists with files. Skipping download (replace=False).


In [4]:
df = pd.read_csv("/".join(["../data/raw/", dataset_name, files[0]]))
df.head()

,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,NaN,2024-07-08 00:17:00,NaN,NaN,0,0,NaN,on_time,on_time
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,50.770202,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,NaN,NaN,0,0,NaN,on_time,on_time
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time


In [ ]:
# Spaltennamen ausgeben

print(df.columns.tolist())



--- VERFÜGBARE SPALTEN ---
['ID', 'line', 'path', 'eva_nr', 'category', 'station', 'state', 'city', 'zip', 'long', 'lat', 'arrival_plan', 'departure_plan', 'arrival_change', 'departure_change', 'arrival_delay_m', 'departure_delay_m', 'info', 'arrival_delay_check', 'departure_delay_check']


In [ ]:
#df.drop(
#    labels=['id', 'line', 'eva_nr', 'state', 'zip', 'long', 'lat'], 
#    axis=1, 
#    inplace=True, 
#    errors='ignore')

In [ ]:
#df['scheduled_departure'] = pd.to_datetime(df['scheduled_departure'], errors='coerce')
#df['actual_departure'] = pd.to_datetime(df['actual_departure'], errors='coerce')
#df['scheduled_arrival'] = pd.to_datetime(df['scheduled_arrival'], errors='coerce')
#df['actual_arrival'] = pd.to_datetime(df['actual_arrival'], errors='coerce')    

In [ ]:
#date_format = "%Y-%m-%d %H:%M:%S"
#df["arrival_plan"] = pd.to_datetime(df["arrival_plan"], format=date_format)
#df["departure_plan"] = pd.to_datetime(df["departure_plan"], format=date_format)
#df["arrival_change"] = pd.to_datetime(df["arrival_change"], format=date_format)
#df["departure_change"] = pd.to_datetime(df["departure_change"], format=date_format)

#df["arrival_plan_time"] = df["arrival_plan"].dt.time
#df["arrival_plan_date"] = df["arrival_plan"].dt.date

#df["departure_plan_time"] = df["departure_plan"].dt.time
#df["departure_plan_date"] = df["departure_plan"].dt.date

In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2061357 entries, 0 to 2061356
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     object 
 1   line                   object 
 2   path                   object 
 3   eva_nr                 int64  
 4   category               int64  
 5   station                object 
 6   state                  object 
 7   city                   object 
 8   zip                    int64  
 9   long                   float64
 10  lat                    float64
 11  arrival_plan           object 
 12  departure_plan         object 
 13  arrival_change         object 
 14  departure_change       object 
 15  arrival_delay_m        int64  
 16  departure_delay_m      int64  
 17  info                   object 
 18  arrival_delay_check    object 
 19  departure_delay_check  object 
dtypes: float64(2), int64(5), object(13)
memory usage: 314.5+ MB


In [9]:
# Funktionen definieren???
def classify_time_of_day(hour):
    if 5 <= hour < 9:
        return 'HVZ_Morgen (05-09 Uhr)'
    elif 9 <= hour < 17:
        return 'Tag (09-17 Uhr)'
    elif 17 <= hour < 20:
        return 'HVZ_Abend (17-20 Uhr)'
    else:
        return 'Nacht/Spät (20-05 Uhr)'

def classify_delay(minutes):
    if minutes <= 5:
        return 'Klein (1-5 Min)'
    elif 5 < minutes <= 15:
        return 'Mittel (6-15 Min)'
    else:
        return 'Schwer (> 15 Min)' 

def load_and_filter_leipzig(df, cols, delay_type):
    """
    Lädt und filtert die DB-Daten für alle Leipziger Stationen basierend auf Ankunft oder Abfahrt.
    """
    try:
        # 1. Spaltenauswahl (WICHTIG: Prüft die Namen aus Phase 1)
        df_temp = df[cols].copy()
        
        # 2. Umbenennen
        df_temp = df_temp.rename(columns={
            cols[1]: 'datetime',
            cols[2]: 'delay_in_minutes'
        })
        
        # 3. Bereinigung der Verspätungsspalte
        # Stellt sicher, dass die Verspätung eine Zahl ist
        df_temp['delay_in_minutes'] = pd.to_numeric(df_temp['delay_in_minutes'], errors='coerce')
        df_temp = df_temp.dropna(subset=['delay_in_minutes', 'datetime'])
        
        # 4. Zeitstempel konvertieren
        df_temp['datetime'] = pd.to_datetime(df_temp['datetime'], errors='coerce')
        df_temp = df_temp.dropna(subset=['datetime'])
        
        # 5. Filtern auf ALLE Leipzig-Stationen und tatsächliche Verspätung (> 0)
        df_temp = df_temp[
            (df_temp['station'].str.contains(LEIPZIG_NAME_FILTER, case=False, na=False)) & 
            (df_temp['delay_in_minutes'] > 0)
        ].copy()
        
        df_temp['delay_type'] = delay_type
        
        return df_temp
    
    except KeyError as e:
        print(f"WARNUNG: Spalte {e} nicht gefunden. {delay_type}-Daten werden übersprungen.")
        return pd.DataFrame() 

In [14]:

# daten zusammenführen
print("\n--- DATEN ZUSAMMENFÜHREN ---")
df_leipzig_arrival = load_and_filter_leipzig(df, COLS_ARRIVAL, 'Ankunft')
df_leipzig_departure = load_and_filter_leipzig(df, COLS_DEPARTURE, 'Abfahrt')

df_leipzig = pd.concat([df_leipzig_arrival, df_leipzig_departure], ignore_index=True)

#
if len(df_leipzig) > 0:
    df_leipzig['hour'] = df_leipzig['datetime'].dt.hour
    df_leipzig['day_of_week'] = df_leipzig['datetime'].dt.day_name()
    df_leipzig['time_segment'] = df_leipzig['hour'].apply(classify_time_of_day)
    df_leipzig['delay_category'] = df_leipzig['delay_in_minutes'].apply(classify_delay)

    print(f"Ankunftsverspätungen (Leipzig gesamt): {len(df_leipzig_arrival):,}")
    print(f"Abfahrtsverspätungen (Leipzig gesamt): {len(df_leipzig_departure):,}")
    print(f"Gesamt-Verspätungsereignisse (Leipzig gesamt): {len(df_leipzig):,}")
    print("\nErste Zeilen des finalen, bereinigten DataFrames:")
    print(df_leipzig[['datetime', 'station', 'delay_type', 'delay_in_minutes', 'time_segment', 'delay_category']].head())
else:
    print("\nFEHLER!")



--- DATEN ZUSAMMENFÜHREN ---
Ankunftsverspätungen (Leipzig gesamt): 4,798
Abfahrtsverspätungen (Leipzig gesamt): 9,054
Gesamt-Verspätungsereignisse (Leipzig gesamt): 13,852

Erste Zeilen des finalen, bereinigten DataFrames:
             datetime                        station delay_type  \
0 2024-07-08 00:08:00  Leipzig-Völkerschlachtdenkmal    Ankunft   
1 2024-07-08 00:17:00                  Leipzig Messe    Ankunft   
2 2024-07-08 00:10:00        Flughafen Leipzig/Halle    Ankunft   
3 2024-07-08 00:52:00             Leipzig-Engelsdorf    Ankunft   
4 2024-07-08 00:12:00                  Leipzig Markt    Ankunft   

   delay_in_minutes            time_segment     delay_category  
0                 1  Nacht/Spät (20-05 Uhr)    Klein (1-5 Min)  
1                 1  Nacht/Spät (20-05 Uhr)    Klein (1-5 Min)  
2                 1  Nacht/Spät (20-05 Uhr)    Klein (1-5 Min)  
3                 8  Nacht/Spät (20-05 Uhr)  Mittel (6-15 Min)  
4                 1  Nacht/Spät (20-05 Uhr)    

In [ ]:
df.head()